In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
!pip install openpyxl

import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
!pip install openpyxl

# ---------------------------
# 1. Define files and their column mappings
# ---------------------------
files_info = {
    "fMRI_Original": {
        "file": "/Users/elizabethkudriavtsev/Downloads/Poldrack_Lab_Documents/Beast/thebeastdataset/BeastfMRIN30.xlsx",
        "subject_col": "SubjID",
        "rt_col": "RT",
        "stop_acc_col": "StopAcc",
        "ssd_col": "SSD",
        "stop_go_col": "Stop1OrNoStop2",
        "go_code": 2,
        "stop_code": 1,
        "go_acc": "GoAcc",
        "stim_col": "Target1NonTarget2TempNonTarget3",
        "stim_map": {1: "Target", 2: "Non-target", 3: "Lure"}
    },
    "FollowUp": {
        "file": "/Users/elizabethkudriavtsev/Downloads/Poldrack_Lab_Documents/Beast/thebeastdataset/N23NonFMRIBeast.xlsx",
        "subject_col": "subject_id",
        "rt_col": "RT",
        "stop_acc_col": "stop_acc",
        "ssd_col": "SSD_target",
        "stop_go_col": "SS_trial_type",
        "go_code": "go",
        "stop_code": "stop",
        "go_acc": "go_acc",
        "stim_col": "adaptive_trial_type",
        "stim_map": {"target": "Target", "non-target": "Non-target", "temp-non-target": "Lure"}
    },
    "1BackStop": {
        "file": "/Users/elizabethkudriavtsev/Downloads/Poldrack_Lab_Documents/Beast/thebeastdataset/1BackStoppingN23.xlsx",
        "subject_col": "subject_id",
        "rt_col": "rt",
        "stop_acc_col": "stop_acc",
        "ssd_col": "SSD_target",
        "stop_go_col": "stop_trial_type",
        "go_code": "go",
        "go_acc": "go_acc",
        "stop_code": "stop",
        "stim_col": "adaptive_trial_type",
        "stim_map": {"target": "Target", "non-target": "Non-target", "temp-non-target": "Lure"}
    },
    "NBackNoStop": {  # no stop trials
        "file": "/Users/elizabethkudriavtsev/Downloads/Poldrack_Lab_Documents/Beast/thebeastdataset/NBackNoStopN23.xlsx",
        "subject_col": "subject_id",
        "rt_col": "rt",
        "stop_acc_col": None,
        "ssd_col": None,
        "stop_go_col": None,
        "go_code": None,
        "stop_code": None,
        "go_acc": None,
        "stim_col": "adaptive_trial_type",
        "stim_map": {"target": "Target", "non-target": "Non-target", "temp-non-target": "Lure"}
    },
    "ShapeStop": {
    "file": "/Users/elizabethkudriavtsev/Downloads/Poldrack_Lab_Documents/Beast/thebeastdataset/StopNoNBackN23.xlsx",
    "subject_col": "subject_ID",
    "rt_col": "rt",
    "stop_acc_col": "stop_acc",
    "ssd_col": "SS_delay",
    "stop_go_col": "SS_trial_type",
    "go_code": "go",     # <-- strings, not 0/1
    "stop_code": "stop",
    "go_acc": "go_acc",
    "shape_col": "go_stim",
    "shape_map": {"black_circle": "Circle", "black_triangle": "Triangle"}
}
}

# ---------------------------
# 2. SSRT computation function
# ---------------------------
def compute_ssrt(df, stop_go_col, stop_acc_col, ssd_col, rt_col, go_code, stop_code):
    df = df.copy()
    
    # skip if no stop trials (e.g., NBackNoStop)
    if stop_go_col is None or stop_acc_col is None or ssd_col is None:
        return np.nan
    
    # Map stop/go to 1 = stop, 2 = go
    df['stop_trial'] = df[stop_go_col].map({stop_code: 1, go_code: 2})
    go_trials = df[df['stop_trial'] == 2]
    stop_trials = df[df['stop_trial'] == 1]
    
    if len(go_trials) == 0 or len(stop_trials) == 0:
        return np.nan
    
    p_respond = 1 - stop_trials[stop_acc_col].mean()
    go_rts = np.sort(go_trials[rt_col].values)
    n_idx = int(np.ceil(p_respond * len(go_rts))) - 1
    n_idx = max(n_idx, 0)
    nth_rt = go_rts[n_idx]
    mean_ssd = stop_trials[ssd_col].mean()
    
    return nth_rt - mean_ssd

# ---------------------------
# 3. Loop through files and compute SSRT
# ---------------------------
ssrt_list = []

for cond_name, info in files_info.items():
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()  # remove whitespace
    
    # Robust subject column detection
    subject_col = next((c for c in df.columns if info['subject_col'].lower() in c.lower()), None)
    if subject_col is None:
        raise ValueError(f"Subject column not found in {cond_name} dataset")
    df.rename(columns={subject_col: 'subject'}, inplace=True)
    
    for subj in df['subject'].unique():
        subj_df = df[df['subject'] == subj]
        try:
            ssrt = compute_ssrt(
                subj_df,
                stop_go_col=info.get('stop_go_col'),
                stop_acc_col=info.get('stop_acc_col'),
                ssd_col=info.get('ssd_col'),
                rt_col=info['rt_col'],
                go_code=info.get('go_code'),
                stop_code=info.get('stop_code')
            )
            ssrt_list.append({"subject": subj, "Condition": cond_name, "SSRT": ssrt})
        except Exception as e:
            print(f"Skipping subject {subj} in {cond_name}: {e}")

ssrt_df = pd.DataFrame(ssrt_list)

# Filter extreme SSRTs
ssrt_df = ssrt_df[(ssrt_df['SSRT'] >= 100) & (ssrt_df['SSRT'] <= 600)]

# ---------------------------
# 4. Plot SSRT across stopping conditions
# ---------------------------
plt.figure(figsize=(10,6))

# Correct order: ShapeStop, 1BackStop, fMRI_Original, FollowUp
condition_order = ["ShapeStop", "1BackStop", "fMRI_Original", "FollowUp"]

# Apply categorical ordering
ssrt_df['Condition'] = pd.Categorical(ssrt_df['Condition'], categories=condition_order, ordered=True)

# Spaghetti plot (each subject)
for sub in ssrt_df['subject'].unique():
    sub_df = ssrt_df[ssrt_df['subject'] == sub].set_index('Condition').reindex(condition_order)
    plt.plot(condition_order, sub_df['SSRT'], color='gray', alpha=0.4)

# Mean line
mean_df = ssrt_df.groupby('Condition')['SSRT'].mean().reindex(condition_order)
plt.plot(condition_order, mean_df, color='black', linewidth=3, marker='o')

# Overlay individual points
for cond in condition_order:
    y_vals = ssrt_df[ssrt_df['Condition'] == cond]['SSRT']
    x_vals = [cond]*len(y_vals)
    plt.scatter(x_vals, y_vals, color='gray', alpha=0.5)

plt.xlabel("Condition")
plt.ylabel("SSRT (ms)")
plt.title("SSRT Across Stopping Conditions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ---------------------------
# SSRT ANALYSIS BY LOAD
# ---------------------------
ssrt_results = []
target_tasks = ["fMRI_Original", "FollowUp"]

for task in target_tasks:
    if task not in files_info: continue
    info = files_info[task]
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()
    
    # 1. Standardize Columns (Consistent with Cell 1)
    subj_col = next((c for c in df.columns if 'subj' in c.lower()), 'subject')
    acc_col = next((c for c in df.columns if 'stop' in c.lower() and 'acc' in c.lower()), None)
    load_col = next((c for c in df.columns if c.lower() == 'load'), 'Load')
    rt_col = info['rt_col']
    ssd_col = info.get('ssd_col')
    
    df.rename(columns={subj_col: 'subject', load_col: 'Load'}, inplace=True)

    # 2. Compute SSRT for each Subject at each Load level
    for (subj, load), subj_load_df in df.groupby(['subject', 'Load']):
        
        # Split into Go and Stop trials
        if 'NoStop1StopSuccess2StopFail3' in df.columns:
            go_trials = subj_load_df[subj_load_df['NoStop1StopSuccess2StopFail3'] == 1]
            stop_trials = subj_load_df[subj_load_df['NoStop1StopSuccess2StopFail3'].isin([2, 3])]
        else:
            go_trials = subj_load_df[subj_load_df['stop_trial_type'] == 'go']
            stop_trials = subj_load_df[subj_load_df['stop_trial_type'] != 'go']

        # Check for sufficient data (need both trial types to compute SSRT)
        if len(go_trials) >= 5 and len(stop_trials) >= 5:
            # Integration Method Calculation
            p_respond = 1 - stop_trials[acc_col].mean()
            go_rts = np.sort(go_trials[rt_col].dropna().values)
            
            # Find the RT corresponding to the probability of responding
            n_idx = int(np.ceil(p_respond * len(go_rts))) - 1
            n_idx = max(0, min(n_idx, len(go_rts) - 1))
            
            ssrt = go_rts[n_idx] - stop_trials[ssd_col].mean()
            
            ssrt_results.append({
                "subject": subj, 
                "Condition": task, 
                "Load": load, 
                "SSRT": ssrt
            })

# 3. Clean and Plot
ssrt_df = pd.DataFrame(ssrt_results)
# Remove outliers (SSRTs outside 50-800ms are usually invalid)
ssrt_df = ssrt_df[(ssrt_df['SSRT'] >= 50) & (ssrt_df['SSRT'] <= 800)]

plt.figure(figsize=(10, 6))
sns.lineplot(data=ssrt_df, x='Load', y='SSRT', hue='Condition', marker='o', errorbar='se')

plt.title("Inhibitory Speed (SSRT) by N-Back Load")
plt.ylabel("SSRT (ms) - Lower is Faster Stopping")
plt.xlabel("N-Back Load")
plt.xticks([1, 2, 3, 4])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print averages for your slides
print("Mean SSRT per Load:")
print(ssrt_df.groupby('Load')['SSRT'].mean().round(2))

In [ ]:
# ---------------------------
# Trial counts per condition x trial type
# ---------------------------

trial_counts_list = []

for task_name, info in files_info.items():
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()

    # Standardize subject column
    df.rename(columns={info['subject_col']: 'subject'}, inplace=True)

    # ---- Stimulus label ----
    if task_name == "ShapeStop":
        df['stim_label'] = df[info['shape_col']].map(info['shape_map'])
    else:
        df['stim_label'] = df[info['stim_col']].map(info['stim_map'])

    # ---- Trial type ----
    if info['stop_go_col'] is not None:
        df['trial_type'] = df[info['stop_go_col']].map({info['stop_code']: 'Stop', info['go_code']: 'Go'})
        df['trial_label'] = df['trial_type'] + ' – ' + df['stim_label']
    else:
        df['trial_type'] = 'Go'
        df['trial_label'] = 'Go – ' + df['stim_label']

    # ---- Count trials per label ----
    counts = df.groupby('trial_label').size().reset_index(name='n_trials')
    counts['task'] = task_name
    trial_counts_list.append(counts)

# Combine all tasks
trial_counts = pd.concat(trial_counts_list, ignore_index=True)

# ---------------------------
# 1. Define trial types to include (for N-back)
# ---------------------------
trial_types_order = [
    "Go – Target", "Go – Non-target", "Go – Lure",
    "Stop – Target", "Stop – Non-target", "Stop – Lure"
]

# N-back tasks
nback_tasks = ["fMRI_Original", "FollowUp", "1BackStop", "NBackNoStop"]

# ---------------------------
# 2. Pivot data for grouped bar plot
# ---------------------------
# Filter only N-back tasks
nback_counts = trial_counts[trial_counts["task"].isin(nback_tasks)]

# Pivot so that rows = trial_label, columns = task
pivot_df = nback_counts.pivot_table(
    index="trial_label",  # <-- corrected column name
    columns="task",
    values="n_trials",
    fill_value=0
)

# Reorder rows to desired trial type order
pivot_df = pivot_df.reindex(trial_types_order)

# ---------------------------
# 3. Plot grouped bar chart
# ---------------------------
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,6))
bar_width = 0.2
x = np.arange(len(trial_types_order))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for i, task in enumerate(nback_tasks):
    ax.bar(x + i*bar_width, pivot_df[task], width=bar_width, label=task, color=colors[i])

ax.set_xticks(x + bar_width*1.5)
ax.set_xticklabels(trial_types_order, rotation=45, ha="right")
ax.set_ylabel("Number of trials")
ax.set_title("Trial Counts Across N-back Task Variants")
ax.legend(title="Condition")
plt.tight_layout()
plt.show()

# ---------------------------
# 4. Plot ShapeStop separately
# ---------------------------
shape_data = trial_counts[trial_counts["task"] == "ShapeStop"].copy()

plt.figure(figsize=(5,5))
plt.bar(shape_data["trial_label"], shape_data["n_trials"], color="#9467bd")
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of trials")
plt.title("Trial Counts: No N-back Shape Discrimination Stop-Signal Task")
plt.tight_layout()
plt.show()








# ---------------------------
# 5. Stop signal proportion
# ---------------------------


stop_props = []

for task, info in files_info.items():
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()

    # Total trials
    total_trials = len(df)

    stop_col = info.get('stop_go_col')
    go_code = info.get('go_code')
    stop_code = info.get('stop_code')

    if stop_col is not None and go_code is not None and stop_code is not None:
        n_stop = (df[stop_col] == stop_code).sum()
        n_go = (df[stop_col] == go_code).sum()
        prop = n_stop / (n_stop + n_go)
    else:
        # If no stop/go column, assume all go trials
        n_stop = 0
        n_go = total_trials
        prop = 0

    stop_props.append({
        "Task": task,
        "n_go": n_go,
        "n_stop": n_stop,
        "stop_signal_proportion": prop
    })

stop_props_df = pd.DataFrame(stop_props)
print(stop_props_df)

plt.figure(figsize=(6,4))
plt.bar(stop_props_df["Task"], stop_props_df["stop_signal_proportion"], color="#444444")
plt.ylabel("Stop-signal proportion")
plt.title("Stop-signal Proportion by Task")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------
# 1. Prepare data for Go RTs
# ---------------------------
rt_list = []

for task, info in files_info.items():
    df = pd.read_excel(info["file"])
    
    # Clean subject column
    subject_col = next((c for c in df.columns if info['subject_col'].lower() in c.lower()), None)
    df.rename(columns={subject_col: 'subject'}, inplace=True)
    
    # Only include tasks with a stop/go column for Go RT selection
    if info.get("stop_go_col") is not None:
        # Filter Go trials and realistic RTs
        df = df[(df[info["rt_col"]] >= 150) & (df[info["rt_col"]] <= 2000)]
        df = df[df[info["stop_go_col"]] == info["go_code"]]
        df["Condition"] = task
        df["RT"] = df[info["rt_col"]]
        rt_list.append(df[["subject", "Condition", "RT"]])
    else:
        # For tasks with no stop/go (ShapeStop), include all RTs within range
        df = df[(df[info["rt_col"]] >= 150) & (df[info["rt_col"]] <= 2000)]
        df["Condition"] = task
        df["RT"] = df[info["rt_col"]]
        rt_list.append(df[["subject", "Condition", "RT"]])

rt_df = pd.concat(rt_list, ignore_index=True)

# ---------------------------
# 2. Split into N-back vs ShapeStop
# ---------------------------
nback_tasks = ["fMRI_Original", "FollowUp", "1BackStop", "NBackNoStop"]
shape_task = ["ShapeStop"]

rt_nback = rt_df[rt_df["Condition"].isin(nback_tasks)]
rt_shape = rt_df[rt_df["Condition"].isin(shape_task)]

# ---------------------------
# 3. Plot Go RT distributions
# ---------------------------
sns.set(style="whitegrid")

# N-back panel
plt.figure(figsize=(10,5))
sns.kdeplot(
    data=rt_nback, x="RT", hue="Condition", fill=True, common_norm=False, alpha=0.5
)
plt.xlim(150, 2000)
plt.xlabel("Go RT (ms)")
plt.ylabel("Density")
plt.title("Go RT Distribution Across N-back Conditions")
plt.show()

# ShapeStop panel
plt.figure(figsize=(6,5))
sns.kdeplot(
    data=rt_shape, x="RT", hue="Condition", fill=True, common_norm=False, alpha=0.5
)
plt.xlim(150, 2000)
plt.xlabel("Go RT (ms)")
plt.ylabel("Density")
plt.title("Go RT Distribution: Shape Stop Task")
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

acc_list = []

for task, info in files_info.items():
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()  # remove whitespace

    # Standardize subject column
    subject_col = next((c for c in df.columns if info['subject_col'].lower() in c.lower()), None)
    df.rename(columns={subject_col: 'subject'}, inplace=True)

    # Filter Go trials if stop/go column exists
    if info.get('stop_go_col') is not None:
        go_df = df[df[info['stop_go_col']] == info['go_code']].copy()
    else:
        go_df = df.copy()  # everything is Go

    # Handle Go accuracy
    go_acc_col = info.get('go_acc')
    if go_acc_col is not None and go_acc_col in df.columns:
        go_df['acc'] = go_df[go_acc_col]
    else:
        # If no go_acc, use adaptive_acc if available
        if 'adaptive_acc' in df.columns:
            go_df['acc'] = go_df['adaptive_acc']
        else:
            # If no accuracy data, set as NaN
            go_df['acc'] = pd.Series([None] * len(go_df))

    # Compute mean accuracy per subject
    mean_acc = go_df.groupby('subject')['acc'].mean().reset_index()
    mean_acc['Condition'] = task
    
    task_avg = mean_acc['acc'].mean()
    print(f"Task: {task:<15} | Subjects: {len(mean_acc):<3} | Mean Acc: {task_avg:.2%}")
    #

    acc_list.append(mean_acc)

# Combine all tasks
acc_df = pd.concat(acc_list, ignore_index=True)

# Compute mean accuracy per condition, keeping NaN values if necessary
mean_acc_df = acc_df.groupby('Condition')['acc'].mean().reset_index()

# Optional: order conditions
condition_order = ["fMRI_Original", "FollowUp", "1BackStop", "NBackNoStop", "ShapeStop"]
mean_acc_df['Condition'] = pd.Categorical(mean_acc_df['Condition'], categories=condition_order, ordered=True)
mean_acc_df = mean_acc_df.sort_values('Condition')

# Plot
plt.figure(figsize=(8,5))
sns.barplot(data=mean_acc_df, x='Condition', y='acc', palette='pastel')
plt.ylim(0,1)
plt.ylabel('Mean Go Accuracy')
plt.title('Go Accuracy Across Task Variants')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------
# 1. Gather SSDs from all tasks
# ---------------------------
ssd_list = []

for task, info in files_info.items():
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()  # remove any weird spaces

    # Standardize subject column
    df.rename(columns={info['subject_col']: 'subject'}, inplace=True)

    # Only include trials that have SSD (skip tasks without stop signals)
    if info.get('ssd_col') is not None:
        df = df[df[info['stop_go_col']] == info['stop_code']]  # only Stop trials
        df['Condition'] = task
        df['SSD'] = df[info['ssd_col']]
        ssd_list.append(df[['subject', 'Condition', 'SSD']])

# Combine all tasks
ssd_df = pd.concat(ssd_list, ignore_index=True)

# ---------------------------
# 2. Plot SSD distributions
# ---------------------------
sns.set(style="whitegrid")
plt.figure(figsize=(10,6))

# Define colors to match conditions
conditions = ssd_df['Condition'].unique()
palette = sns.color_palette("tab10", n_colors=len(conditions))
color_map = dict(zip(conditions, palette))

# KDE per task
sns.kdeplot(
    data=ssd_df,
    x='SSD',
    hue='Condition',
    fill=True,
    common_norm=False,
    alpha=0.5,
    palette=color_map
)

# Overlay mean SSD as vertical lines with matching colors
mean_ssd = ssd_df.groupby('Condition')['SSD'].mean()
for cond, mean_val in mean_ssd.items():
    plt.axvline(mean_val, linestyle='--', color=color_map[cond], label=f'{cond} mean')

plt.xlabel("SSD (ms)")
plt.ylabel("Density")
plt.title("SSD Distributions Across Stop-Signal Task Variants")
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# List to collect stop trial performance
stop_prob_list = []

for task, info in files_info.items():
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()  # clean column names

    # Standardize subject column
    df.rename(columns={info['subject_col']: 'subject'}, inplace=True)

    # Only process tasks with stop trials
    if info.get('stop_go_col') is not None:
        # Identify stop trials
        stop_trials = df[df[info['stop_go_col']] == info['stop_code']].copy()
        
        # Probability of responding on stop trial: 1 = responded, 0 = successfully inhibited
        # If StopAcc column is 1 = successful stop, 0 = failed stop
        # Then responding = 1 - StopAcc
        stop_trials['responded'] = 1 - stop_trials[info['stop_acc_col']]

        # Compute mean per subject
        subj_mean = stop_trials.groupby('subject')['responded'].mean().reset_index()
        subj_mean['task'] = task
        stop_prob_list.append(subj_mean)

# Combine all tasks
stop_prob_df = pd.concat(stop_prob_list, ignore_index=True)

# ---------------------------
# Plot probability of responding given a stop signal
# ---------------------------
plt.figure(figsize=(8,5))
sns.boxplot(data=stop_prob_df, x='task', y='responded', palette="Set2")
sns.stripplot(data=stop_prob_df, x='task', y='responded', color='black', alpha=0.5, jitter=True)
plt.ylabel("Probability of Responding on Stop Trials")
plt.ylim(0,1)
plt.title("p(respond | stop) Across Tasks")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:

# ---------------------------
# 1. DATA LOADING & CLEANING
# ---------------------------
load_tasks = ["fMRI_Original", "FollowUp"]
all_go_trials = []

# Assuming files_info is defined with your file paths
for task in load_tasks:
    if task not in files_info: continue
    info = files_info[task]
    df = pd.read_excel(info['file'])
    df.columns = df.columns.str.strip()
    
    # --- Standardize Subject ---
    subject_col = next((c for c in df.columns if 'subj' in c.lower()), None)
    df.rename(columns={subject_col: 'subject'}, inplace=True)
    
    # --- Standardize Accuracy (Handles 'GoAcc' and 'go_acc') ---
    acc_col = next((c for c in df.columns if 'go' in c.lower() and 'acc' in c.lower()), None)
    if acc_col:
        df.rename(columns={acc_col: 'acc'}, inplace=True)
    
    # --- Standardize Load ---
    load_col = next((c for c in df.columns if c.lower() == 'load'), None)
    if load_col:
        df['Load'] = df[load_col]

    # --- Standardize Outcome (Fixes the Regressor 9 ValueError) ---
    if 'stop_trial_type' in df.columns:
        df['n_back_outcome'] = df['stop_trial_type'].replace({'go': 'Go'})
    elif 'NoStop1StopSuccess2StopFail3' in df.columns:
        outcome_map = {1: 'Go', 2: 'Stop Success', 3: 'Stop Failure'}
        df['n_back_outcome'] = df['NoStop1StopSuccess2StopFail3'].map(outcome_map)
    else:
        df['n_back_outcome'] = 'Go'

    # --- Filter for Go Trials ---
    if 'NoStop1StopSuccess2StopFail3' in df.columns:
         go_df = df[df['NoStop1StopSuccess2StopFail3'] == 1].copy()
    elif info.get('stop_go_col'):
        go_df = df[df[info['stop_go_col']] == info['go_code']].copy()
    else:
        go_df = df.copy()

    go_df['Condition'] = task
    all_go_trials.append(go_df)

load_df = pd.concat(all_go_trials, ignore_index=True)

# ---------------------------------------------------------
# 2. COHORT IDENTIFICATION (The "Slide 11" 16-Subject Logic)
# ---------------------------------------------------------
subj_load_sets = load_df.groupby('subject')['Load'].unique().apply(set)

def get_balanced_cohort(all_data, required_loads):
    req_set = set(required_loads)
    mask = subj_load_sets.apply(lambda x: req_set.issubset(x))
    valid_subjects = subj_load_sets[mask].index
    return all_data[all_data['subject'].isin(valid_subjects) & all_data['Load'].isin(required_loads)]

# Creating the balanced subgroups
df_12   = get_balanced_cohort(load_df, [1, 2])
df_123   = get_balanced_cohort(load_df, [1, 2, 3])
df_1234 = get_balanced_cohort(load_df, [1, 2, 3, 4])

# ---------------------------
# 3. PLOTTING
# ---------------------------

# === PLOT 1: Go RT Distributions (Balanced Cohorts) ===
def plot_cohort_dist(data, loads, title):
    plt.figure(figsize=(10, 5))
    n_subj = data['subject'].nunique()
    # Using a distinct color palette for the different loads
    colors = sns.color_palette("husl", len(loads))
    
    for i, load in enumerate(loads):
        subset = data[data['Load'] == load]
        # Plot the KDE (Density)
        sns.kdeplot(data=subset, x='RT', fill=True, alpha=0.3, color=colors[i], label=f"Load {load}")
        # Add a vertical dashed line for the mean RT
        plt.axvline(subset['RT'].mean(), color=colors[i], linestyle='--', linewidth=2)
    
    plt.title(f"{title}\n(Analyzing only the {n_subj} subjects who completed all loads)")
    plt.xlabel("Go Reaction Time (ms)")
    plt.xlim(150, 1500)
    plt.legend(title="Difficulty")
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

# Generate the two key comparisons for your slides
plot_cohort_dist(df_12, [1, 2], "Reaction Time Shift: Loads 1 vs 2")
plot_cohort_dist(df_123, [1, 2, 3], "Reaction Time Shift: Loads 1 through 3")
plot_cohort_dist(df_1234, [1, 2, 3, 4], "Reaction Time Shift: Loads 1 through 4")

# === PLOT 2: Mean Go Accuracy by Load (Population View) ===
plt.figure(figsize=(8,5))
sns.barplot(data=load_df, x='Load', y='acc', hue='Condition', palette='magma')
plt.ylabel("Mean Go Accuracy")
plt.ylim(0, 1.05)
plt.title("Mean Go Accuracy by N-back Load (All Subjects)")
plt.show()

# === PLOT 3: Regressor 9 - Encoding Interference ===
plt.figure(figsize=(10,6))
sns.pointplot(
    data=load_df, x='Load', y='acc', hue='n_back_outcome',
    hue_order=['Go', 'Stop Success', 'Stop Failure'],
    palette={'Go': 'gray', 'Stop Success': 'green', 'Stop Failure': 'red'},
    dodge=True, markers=['o', 's', 'X'], capsize=0.1
)
plt.title("Impact of Trial N-Back on Current Go Accuracy\n(Encoding Interference)")
plt.ylim(0.5, 1.05)
plt.show()

# Final Summary for your notes
print(f"Final Counts:")
print(f"- Total unique subjects: {load_df['subject'].nunique()}")
print(f"- Subjects in 1-4 Balanced Cohort: {df_1234['subject'].nunique()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Filter data for 1-back and 2-back loads
one_back_df = load_df[load_df['Load'] == 1]
two_back_df = load_df[load_df['Load'] == 2]

# Identify subjects who have data for both 1-back and 2-back
subjects_1_back = set(one_back_df['subject'].unique())
subjects_2_back = set(two_back_df['subject'].unique())
common_subjects = list(subjects_1_back.intersection(subjects_2_back))

# Extract accuracy data for these common subjects
one_back_acc = one_back_df[one_back_df['subject'].isin(common_subjects)].groupby('subject')['acc'].mean()
two_back_acc = two_back_df[two_back_df['subject'].isin(common_subjects)].groupby('subject')['acc'].mean()

# Combine the data into a single DataFrame for plotting
accuracy_df = pd.DataFrame({
    'Subject': common_subjects,
    '1-back Accuracy': one_back_acc.loc[common_subjects].values,
    '2-back Accuracy': two_back_acc.loc[common_subjects].values
})

# Melt the DataFrame for a paired bar plot
melted_df = accuracy_df.melt(id_vars='Subject', value_vars=['1-back Accuracy', '2-back Accuracy'],
                             var_name='Load', value_name='Accuracy')

# Plot paired bar plot
plt.figure(figsize=(12, 6))
sns.barplot(data=melted_df, x='Subject', y='Accuracy', hue='Load', palette='pastel')

plt.xlabel('Subject')
plt.ylabel('Accuracy')
plt.title('Paired Go Accuracy: 1-back vs 2-back')
plt.xticks(rotation=90)
plt.legend(title='Load')
plt.tight_layout()
plt.show()


###Line pk

# Calculate mean accuracy for each load
mean_accuracy = load_df.groupby('Load')['acc'].mean().reset_index()

# Calculate standard error for each load
se_accuracy = load_df.groupby('Load')['acc'].sem().reset_index()

# Merge the mean and standard error data
summary_df = pd.merge(mean_accuracy, se_accuracy, on='Load', suffixes=('_mean', '_se'))

# Plot the line graph with error bars
plt.figure(figsize=(10, 6))
sns.lineplot(data=summary_df, x='Load', y='acc_mean', marker='o', color='blue')

# Add error bars
plt.errorbar(summary_df['Load'], summary_df['acc_mean'], yerr=summary_df['acc_se'], fmt='o', color='blue', capsize=5)

plt.xlabel('N-back Load')
plt.ylabel('Mean Go Accuracy')
plt.title('Mean Go Accuracy by N-back Load')
plt.xticks(summary_df['Load'])
plt.ylim(0, 1)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# 1. RE-LOAD DATA (Standardizing and Filtering)
# ---------------------------------------------------------
df_list = []  # list to store dataframes for each task

for task in load_tasks: # Loop through every task file
    if task not in files_info: continue # Skip if the file information is missing
    df = pd.read_excel(files_info[task]['file']) 
    df.columns = df.columns.str.strip() # formatting-remove any accidental spaces in column names
    
    # Identify the subject column regardless of capitalization
    sub_col = next((c for c in df.columns if 'subj' in c.lower()), None)
    if sub_col: df.rename(columns={sub_col: 'subject'}, inplace=True) # rename to 'subject'
        
    # Identify the Load column
    load_col = next((c for c in df.columns if c.lower() == 'load'), None)
    if load_col: df['Load'] = df[load_col] # Ensure we have a standardized 'Load' column
    
    #Only keep Load=2 rows and the 16 subjects who had completed n=1 and n=2
    df_filtered = df[(df['Load'] == 2) & (df['subject'].isin(common_subjects))].copy()
    df_list.append(df_filtered) # Add this specific task's data to list

# Combine all task data into one giant table for Load 2
full_load2 = pd.concat(df_list, ignore_index=True)

# ---------------------------------------------------------
# 2. Sequence analysis to reach back "n" trials
# ---------------------------------------------------------

# Reach back 1 trial: What was the trial type immediately before this one?
full_load2['n1_type'] = full_load2.groupby('subject')['NoStop1StopSuccess2StopFail3'].shift(1)
# Reach back 2 trials: What was the trial type 2 steps ago?
full_load2['n2_type'] = full_load2.groupby('subject')['NoStop1StopSuccess2StopFail3'].shift(2)

# Filter to only look at Go trials (Type 1) as "target"
df_go = full_load2[full_load2['NoStop1StopSuccess2StopFail3'] == 1].copy()

# A. CALCULATE DRAG (n-2): 

# Filter for trials where the immediate past (n-1) was a normal Go (1). 
# Then group by what happened 2 trials ago (n-2) and find average Reaction Time.
drag_results_n2 = df_go[df_go['n1_type'] == 1].groupby('n2_type')['RT'].mean()

# B. CALCULATE CAUTION (n-1): 
# Filter for trials where 2 steps ago (n-2) was a normal Go (1).
# Then group by what happened 1 trial ago (n-1) and find average Reaction Time.
caution_results_n1 = df_go[df_go['n2_type'] == 1].groupby('n1_type')['RT'].mean()

# ---------------------------------------------------------
# 3. CALCULATE DIFFERENCES
# ---------------------------------------------------------
# Formula: (Average RT after Stop) minus (Average RT after Go baseline)
# .get(2, drag_results_n2[1]) = "Get the Success mean, but use the Go mean as a backup if empty"
drag_vals = [drag_results_n2.get(2, drag_results_n2[1]) - drag_results_n2[1], drag_results_n2.get(3, drag_results_n2[1]) - drag_results_n2[1]]
caus_vals = [caution_results_n1.get(2, caution_results_n1[1]) - caution_results_n1[1], caution_results_n1.get(3, caution_results_n1[1]) - caution_results_n1[1]]
all_vals = drag_vals + caus_vals # Combine all 4 results for the 4 bars in the plot
print(all_vals)

labels = ['Success (n-2)\nDRAG', 'Failure (n-2)\nDRAG', 
          'Success (n-1)\nCAUTION', 'Failure (n-1)\nCAUTION']

plt.figure(figsize=(10, 6))
colors = ['firebrick', 'lightcoral', 'royalblue', 'lightskyblue']
bars = plt.bar(labels, all_vals, color=colors, edgecolor='black', alpha=0.8)
plt.axhline(0, color='black', linewidth=1)

for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, y + (5 if y >= 0 else -15), 
             f'+{y:.1f} ms' if y > 0 else f'{y:.1f} ms', ha='center', fontweight='bold')

plt.title('Disentangling Drag vs. Caution (Load 2 Comparison)', fontsize=14, fontweight='bold')
plt.ylabel('Change in RT (ms)')
plt.tight_layout()
plt.show()

In [ ]:


#  RE-LOAD DATA (Standardizing and Filtering) for load 3
# ----------------------------------------------------------------------------------

df_list = [] 
for task in ["fMRI_Original", "FollowUp"]: 
    if task not in files_info: continue
    
    # Load raw excel
    df_raw = pd.read_excel(files_info[task]['file'])
    
    # Clean column names (removes accidental leading/trailing spaces from headers)
    df_raw.columns = df_raw.columns.str.strip()
    
    # STANDARDIZE SUBJECT: Finds 'subj', 'Subject', 'subject' etc.
    sub_col = next((c for c in df_raw.columns if 'subj' in c.lower()), None)
    if sub_col: df_raw.rename(columns={sub_col: 'subject'}, inplace=True)
    
    # STANDARDIZE TRIAL ID: Finds 'trial_num', 'TrialNum', 'trialnum' etc.
    # We need this "Anchor" to glue the stimulus rows to the response rows.
    tn_col = next((c for c in df_raw.columns if 'trial' in c.lower() and 'num' in c.lower()), None)
    if tn_col: df_raw.rename(columns={tn_col: 'TrialNum'}, inplace=True)
    
    # STANDARDIZE LOAD: Ensures we have a consistent 'Load' column for filtering
    load_col = next((c for c in df_raw.columns if c.lower() == 'load'), None)
    if load_col: df_raw['Load'] = df_raw[load_col]
    
    # FILTER: Only keep Load 3 rows for this specific analysis
    df_list.append(df_raw[df_raw['Load'] == 3].copy())

# Combine all files into one master table
full_load3 = pd.concat(df_list, ignore_index=True, sort=False)

# ----------------------------------------------------------------------------------
# 2. ACCURACY CONSOLIDATION & ROW ALIGNMENT ("THE GLUE")
# ----------------------------------------------------------------------------------
# Merge all possible accuracy columns into one 'combined_acc'
potential_acc_cols = ['GoAcc', 'go_acc']
full_load3['combined_acc'] = np.nan

for col in potential_acc_cols:
    if col in full_load3.columns:
        # Convert to numeric (handles strings) and fill gaps
        full_load3['combined_acc'] = full_load3['combined_acc'].fillna(pd.to_numeric(full_load3[col], errors='coerce'))

# 3. SEQUENCE ANALYSIS (REACHING BACK 3 TRIALS)
# ----------------------------------------------------------------------------------
# Identify trial types for the 3 trials preceding the current one
full_load3['n1_type'] = full_load3.groupby('subject')['NoStop1StopSuccess2StopFail3'].shift(1)
full_load3['n2_type'] = full_load3.groupby('subject')['NoStop1StopSuccess2StopFail3'].shift(2)
full_load3['n3_type'] = full_load3.groupby('subject')['NoStop1StopSuccess2StopFail3'].shift(3)

# Filter for the "Clean" Sequence:
# 1. Current trial must be a 'Go' (Type 1)
# 2. n-1 and n-2 must also be 'Go' (Type 1) to isolate the 'n-3' effect.
df_go = full_load3[full_load3['NoStop1StopSuccess2StopFail3'] == 1].copy()
df_drag3 = df_go[(df_go['n1_type'] == 1) & (df_go['n2_type'] == 1)].copy()

# ----------------------------------------------------------------------------------
# 4. FINAL CALCULATION & OUTPUT
# ----------------------------------------------------------------------------------
# RT Statistics
drag_results_rt = df_drag3.groupby('n3_type')['RT'].mean()
base_rt = drag_results_rt.get(1)

# Accuracy Statistics
acc_results = df_drag3.groupby('n3_type')['combined_acc'].mean()
base_acc = acc_results.get(1)

# Difference Math for Stop Failure
rt_diff = (drag_results_rt.get(3) - base_rt) if 3 in drag_results_rt.index else 0
acc_diff = (acc_results.get(3) - base_acc) if 3 in acc_results.index else 0

print("--- LOAD 3 ANALYSIS: THE POST-FAILURE 'LOCK-IN' ---")
print(f"Speeding Effect (RT): {rt_diff:.1f} ms faster than baseline")
print(f"Accuracy Change:      {acc_diff*100:+.1f}% improvement")

# ----------------------------------------------------------------------------------
# 5. VISUALIZATION
# ----------------------------------------------------------------------------------
labels = ['Stop Success (n-3)', 'Stop Failure (n-3)']
vals = [(drag_results_rt.get(2) - base_rt) if 2 in drag_results_rt.index else 0, rt_diff]

plt.figure(figsize=(8, 6))
colors = ['firebrick', 'lightcoral']
bars = plt.bar(labels, vals, color=colors, edgecolor='black', alpha=0.8)
plt.axhline(0, color='black', linewidth=1.5)

for bar in bars:
    y = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, y + (5 if y >= 0 else -15), 
             f'{y:.1f} ms', ha='center', fontweight='bold', fontsize=12)

plt.title('Load 3: Change in Reaction Time (n-3 Drag)', fontweight='bold')
plt.ylabel('Change relative to Go Baseline (ms)')
plt.show()
print(f"Post-Failure: {acc_results_n3.get(3)*100:.1f}%")


# ---------------------------------------------------------
# FINAL SUMMARY TABLE
# ---------------------------------------------------------
summary_data = {
    'Metric': ['Reaction Time (RT)', 'Choice Accuracy'],
    'Go Baseline (n-3)': [f"{base_rt:.1f} ms", f"{base_acc*100:.1f}%"],
    'Post-Stop Failure (n-3)': [f"{drag_results_rt.get(3):.1f} ms", f"{acc_results.get(3)*100:.1f}%"],
    'Net Change': [f"{rt_diff:.1f} ms", f"{acc_diff*100:+.1f}%"]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*50)
print("             LOAD 3 PERFORMANCE SUMMARY")
print("="*50)
print(summary_df.to_string(index=False))
print("="*50)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Use the ALREADY standardized load_df
subject_stats = load_df.groupby(['subject', 'Load'])[['RT', 'acc']].mean().reset_index()

# --- NEW: Calculate Total Group Percentages per Load ---
# This calculates the overall mean across all participants for each load level
group_summary = load_df.groupby('Load')['acc'].mean().reset_index()
group_summary['acc_pct'] = group_summary['acc'] * 100 # Convert to percentage for easy reading

# 2. Setup the figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))

def high_vis_plot(data, y_col, title, ax):
    if data.empty: 
        print(f"No data found for {y_col}")
        return

    norm = plt.Normalize(data[y_col].min(), data[y_col].max())
    cmap = plt.cm.RdYlGn_r if y_col == 'RT' else plt.cm.RdYlGn
    
    for subj in data['subject'].unique():
        subj_data = data[data['subject'] == subj].sort_values('Load')
        last_val = subj_data[y_col].iloc[-1]
        color = cmap(norm(last_val))
        ax.plot(subj_data['Load'], subj_data[y_col], marker='o', color=color, alpha=0.3, linewidth=1.5)
        last_load = subj_data['Load'].iloc[-1]
        ax.text(last_load + 0.05, last_val, f"{int(subj)}", fontsize=8, color=color, va='center')

    # Add the "Group Average" - Thick black line
    avg_data = data.groupby('Load')[y_col].mean().reset_index()
    ax.plot(avg_data['Load'], avg_data[y_col], color='black', linewidth=5, marker='s', markersize=10, label='GROUP AVG', zorder=100)
    
    # Add Text Labels for the Group Averages so you can see the % on the chart
    for i, row in avg_data.iterrows():
        val_display = f"{row[y_col]:.1f}ms" if y_col == 'RT' else f"{row[y_col]*100:.1f}%"
        ax.text(row['Load'], row[y_col] + (0.02 if y_col == 'acc' else 10), 
                val_display, color='black', fontweight='bold', ha='center', fontsize=12)

    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.set_xticks([1, 2, 3])
    ax.set_xlabel('N-Back Load', fontsize=12)
    ax.set_ylabel('Mean RT (ms)' if y_col == 'RT' else 'Accuracy', fontsize=12)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xlim(0.8, 3.6)

# Run the plotting functions
high_vis_plot(subject_stats, 'RT', 'Individual Go RT Trajectories', ax1)
high_vis_plot(subject_stats, 'acc', 'Individual Go Accuracy Trajectories', ax2)

plt.tight_layout()
plt.show()

# --- PRINT SUMMARY FOR MENTOR ---
print("--- TOTAL GROUP GO ACCURACY BY LOAD ---")
for _, row in group_summary.iterrows():
    print(f"Load {int(row['Load'])}: {row['acc_pct']:.2f}%")

# --- BAR PLOTS ---
plt.figure(figsize=(20, 7))
sns.barplot(data=subject_stats, x='subject', y='acc', hue='Load', palette='magma')
plt.title('Mean Go Accuracy by Subject and Load', fontsize=16, fontweight='bold')
plt.axhline(group_summary[group_summary['Load']==3]['acc'].values[0], 
            color='red', linestyle='--', label='Load 3 Group Mean')
plt.legend()
plt.show()